# The Need for Representation: A Sentiment Analysis Perspective

To make a machine understand text, we must bridge a fundamental gap: Machine learning algorithms (like neural networks or logistic regression) optimize over continuous mathematical spaces, but **text is discrete and symbolic**.


# $f(\text{text}) \rightarrow \mathbf{x} \in \mathbb{R}^d$

## The Challenge

Consider these three movie reviews:

1.   "The movie was excellent." (Positive)
2.   "The film was excellent." (Positive)
3.   "The movie was terrible." (Negative)

To a human, Review 1 and Review 2 are nearly identical in meaning. To a computer seeing raw strings, "movie" and "film" are as different as "movie" and "terrible." Our goal in **Natural Language Representation** is to find a vector space where sentences with similar meanings (or sentiments) are mathematically close to one another.

In [2]:
# Let's set up our baseline corpus in Python.

# Our toy dataset designed to expose representation flaws
corpus = [
    "The movie was excellent",      # Doc 0: Positive
    "The film was excellent",       # Doc 1: Positive (Synonym used)
    "The movie was terrible",       # Doc 2: Negative
    "A terrible terrible movie"     # Doc 3: Negative (Repetition)
]

labels = [1, 1, 0, 0] # 1: Positive, 0: Negative

## Term-Document Matrix (Bag of Words)

The most intuitive way to represent a document is by simply counting the words inside it. This is known as the Bag of Words (BoW) model, which produces a Term-Document Matrix.

### How it Works


1.   **Vocabulary Creation:** Extract all unique words from the entire corpus. This becomes our feature space $\mathbb{R}^V$, where $V$ is the vocabulary size (unique words in the corpus).
2.   **Counting:** For each document, count the occurrences of each word.

Mathematically, a document $d_i$ is represented as a vector $\mathbf{x}_i$, where each element $x_{i,j}$ represents the frequency of term $t_j$:

# $x_{i,j} = \text{count}(t_j, d_i)$


### Conceptual Flow

| Document | a | excellent | film | movie | terrible | the | was |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| 1. "The movie was excellent" | 0 | 1 | 0 | 1 | 0 | 1 | 1 |
| 2. "The film was excellent" | 0 | 1 | 1 | 0 | 0 | 1 | 1 |
| 3. "The movie was terrible" | 0 | 0 | 0 | 1 | 1 | 1 | 1 |
| 4. "A terrible terrible movie" | 1 | 0 | 0 | 1 | 2 | 0 | 0 |


### Fatal Flaws for Sentiment Analysis
While simple, the **Term-Document Matrix** introduces three major problems that require better representations:



1.   **The Synonym Problem (Orthogonality):** "movie" and "film" occupy entirely different columns. The vectors for Doc 1 and Doc 2 are mathematically pushed apart, even though they mean the same thing.
2.   **The Stop-Word Problem:** Highly frequent but meaningless words (like "the", "was", "a") dominate the vector magnitude, overshadowing the actual sentiment-bearing words ("excellent", "terrible").
3. **Extreme Sparsity:** In a real-world vocabulary of 50,000 words, a 20-word sentence will be a vector of 49,980 zeros. This consumes massive memory and makes neural networks prone to overfitting.

In [3]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
# 1. Initialize the CountVectorizer
count_vectorizer = CountVectorizer()

# 2. Fit and transform the corpus into a Term-Document Matrix
tdm = count_vectorizer.fit_transform(corpus)

# Convert to a dense Pandas DataFrame for visualization
vocab = count_vectorizer.get_feature_names_out()
df_tdm = pd.DataFrame(tdm.toarray(), columns=vocab, index=[f"Doc {i}" for i in range(len(corpus))])

print("--- Vocabulary ---")
print(vocab)
print("\n")

print("--- Term-Document Matrix ---")
display(df_tdm)
print("\n")

# Let's check how similar Doc 0 ("The movie was excellent") is to Doc 1 ("The film was excellent")
# Doc 0 ("The movie was excellent") vs Doc 2 ("The movie was terrible")

sim_matrix = cosine_similarity(tdm)

print("\n--- Cosine Similarities ---")
print(f"Doc 0 and Doc 1 ('movie' vs 'film'): {sim_matrix[0, 1]:.2f}")
print(f"Doc 0 and Doc 2 ('excellent' vs 'terrible'): {sim_matrix[0, 2]:.2f}")

# Notice how Doc 0 is mathematically equally similar to Doc 1 and Doc 2 (0.75),
# completely failing to capture that Doc 0 and 1 share the same sentiment/meaning!

--- Vocabulary ---
['excellent' 'film' 'movie' 'terrible' 'the' 'was']


--- Term-Document Matrix ---


,excellent,film,movie,terrible,the,was
Doc 0,1,0,1,0,1,1
Doc 1,1,1,0,0,1,1
Doc 2,0,0,1,1,1,1
Doc 3,0,0,1,2,0,0





--- Cosine Similarities ---
Doc 0 and Doc 1 ('movie' vs 'film'): 0.75
Doc 0 and Doc 2 ('excellent' vs 'terrible'): 0.75


In [8]:
# To find the similarity between words, we transpose the matrix (tdm.T)
# so words become the rows and documents become the columns.
word_sim_matrix = cosine_similarity(tdm.T)
df_word_sim = pd.DataFrame(word_sim_matrix, index=vocab, columns=vocab)

# Extract the specific similarity between 'movie' and 'film'
sim_movie_film = df_word_sim.loc['movie', 'film']

print("\n--- Word Similarities ---")
print(f"Similarity between 'movie' and 'film': {sim_movie_film:.4f}")
print("Vector for 'movie':", df_tdm['movie'].values)
print("Vector for 'film': ", df_tdm['film'].values)


--- Word Similarities ---
Similarity between 'movie' and 'film': 0.0000
Vector for 'movie': [1 0 1 1]
Vector for 'film':  [0 1 0 0]


**Why is the similarity 0.0? (The Synonym Problem)**

Look at the vectors printed above.

* The vector for "movie" across the 4 documents is [1, 0, 1, 1]
* The vector for "film" across the 4 documents is [0, 1, 0, 0]

Because they never appear in the exact same document in our tiny corpus, their dot product is 0. They are perfectly orthogonal. In a sparse Term-Document representation, the model has absolutely no idea that "movie" and "film" mean the same thing.

## Term Frequency - Inverse Document Frequency (TF-IDF)

As we saw with the Term-Document Matrix, representing text by simple counts introduces the Stop-Word Problem. Highly frequent but meaningless words (like "the" and "was") can dominate the vector magnitude, overshadowing the actual sentiment-bearing words (like "excellent" and "terrible").

**TF-IDF** is designed to solve this specific issue by penalizing words that appear too frequently across the entire dataset.

### The Math

1. **Term Frequency (TF):** Measures how frequently a term appears in a document.
# $TF(t, d) = \frac{\text{Count of term } t \text{ in document } d}{\text{Total words in document } d}$

2. **Inverse Document Frequency (IDF):** Measures how important a term is. It scales up rare words and scales down common words across the entire corpus.
# $IDF(t, D) = \log \left( \frac{N}{1 + |\{d \in D : t \in d\}|} \right)$
*(Note: $N$ is the total number of documents. The $+1$ in the denominator prevents division by zero).*

3. **TF-IDF Score:** The final weight of the word in the document.
# $TF\text{-}IDF(t, d, D) = TF(t, d) \times IDF(t, D)$

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [11]:
# 1. Initialize the TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer()

# 2. Fit and transform the corpus into a TF-IDF Matrix
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus)

# Convert to a dense Pandas DataFrame for visualization
df_tfidf = pd.DataFrame(tfidf_matrix.toarray(), columns=vocab, index=[f"Doc {i}" for i in range(len(corpus))])

print("--- TF-IDF Matrix ---")
display(df_tfidf.round(3))
print("\n")

--- TF-IDF Matrix ---


,excellent,film,movie,terrible,the,was
Doc 0,0.581,0.000,0.470,0.000,0.470,0.470
Doc 1,0.505,0.641,0.000,0.000,0.409,0.409
Doc 2,0.000,0.000,0.470,0.581,0.470,0.470
Doc 3,0.000,0.000,0.375,0.927,0.000,0.000


In [12]:
# ---------------------------------------------------------
# Document-Level Similarity (Rows)
# ---------------------------------------------------------
doc_sim_matrix_tfidf = cosine_similarity(tfidf_matrix)

print("--- Document Similarities (TF-IDF) ---")
print(f"Doc 0 and Doc 1 ('movie' vs 'film'): {doc_sim_matrix_tfidf[0, 1]:.2f}")
print(f"Doc 0 and Doc 2 ('excellent' vs 'terrible'): {doc_sim_matrix_tfidf[0, 2]:.2f}")

# ---------------------------------------------------------
# Word-Level Similarity (Columns)
# ---------------------------------------------------------
# Transpose the TF-IDF matrix to check word similarities
word_sim_matrix_tfidf = cosine_similarity(tfidf_matrix.T)
df_word_sim_tfidf = pd.DataFrame(word_sim_matrix_tfidf, index=vocab, columns=vocab)

# Extract the specific similarity between 'movie' and 'film'
sim_movie_film_tfidf = df_word_sim_tfidf.loc['movie', 'film']

print("\n--- Word Similarities (TF-IDF) ---")
print(f"Similarity between 'movie' and 'film': {sim_movie_film_tfidf:.4f}")
print("TF-IDF Vector for 'movie':", df_tfidf['movie'].values.round(3))
print("TF-IDF Vector for 'film': ", df_tfidf['film'].values.round(3))

--- Document Similarities (TF-IDF) ---
Doc 0 and Doc 1 ('movie' vs 'film'): 0.68
Doc 0 and Doc 2 ('excellent' vs 'terrible'): 0.66

--- Word Similarities (TF-IDF) ---
Similarity between 'movie' and 'film': 0.0000
TF-IDF Vector for 'movie': [0.47  0.    0.47  0.375]
TF-IDF Vector for 'film':  [0.    0.641 0.    0.   ]


### What did we solve?
*   **The Stop-Word Problem:** Look at the TF-IDF Matrix. The words "the" and "was" now have lower numerical weights compared to the rarer, sentiment-bearing words like "terrible" or "excellent".

### What problems still persist?
Despite better weighting, our core architectural flaws remain:

1.  **The Synonym / Word Meaning Problem:** The similarity between "movie" and "film" is still **0.0000**. Because they never appear in the exact same document in our tiny corpus, their dot product is 0. They are still treated as perfectly orthogonal dimensions. The model still has absolutely no idea that "movie" and "film" mean the same thing.
2.  **Extreme Sparsity:** Our vectors are still the size of the entire vocabulary ($V$). In a real-world vocabulary of 50,000 words, our vectors will still be filled with 49,980 zeros, just with slightly different non-zero values.
3.  **Loss of Sequence and Order:** Because this is still a "Bag of Words" approach, it destroys sentence structure. The phrase *"good, not bad"* would have the exact same mathematical representation as *"bad, not good"*.

To solve the Synonym and Sparsity problems, we must abandon the idea that "one word = one dimension" and move to **Dense Word Embeddings**.

## 3. Word Embeddings (Word2Vec)

As we saw in the previous sections, TF-IDF and the Term-Document Matrix suffer from two fatal flaws: Extreme Sparsity and the Synonym Problem[cite: 2]. To solve these issues, we must abandon the "one word = one dimension" paradigm.

Instead, we map words to a dense, lower-dimensional continuous vector space $\mathbb{R}^d$ (where $d$ is typically 50 to 300). This approach is grounded in the **Distributional Hypothesis** (J.R. Firth, 1957):
> *"You shall know a word by the company it keeps."*

If two words (like "movie" and "film") appear in similar contexts, they should have mathematically similar vector representations.

### Architectures: CBOW vs. Skip-Gram

Word2Vec (Mikolov et al., 2013) is trained on a massive text corpus using one of two self-supervised objectives:

<p align="center">
  <img src="https://d2l.ai/_images/cbow.svg" alt="CBOW Architecture" width="40%">
  &nbsp; &nbsp; &nbsp; &nbsp;
  <img src="https://d2l.ai/_images/skip-gram.svg" alt="Skip-Gram Architecture" width="40%">
</p>
<p align="center"><em>(Left: CBOW predicts the target word from its context. Right: Skip-Gram predicts the context from a target word. Images via d2l.ai)</em></p>

*   **Continuous Bag-of-Words (CBOW):** Tries to predict the target word given its surrounding context words. Best for faster training and representing frequent words accurately.
*   **Skip-Gram:** Uses the target word to predict the surrounding context words. Best for smaller datasets and capturing the meaning of rare words.

### Under the Hood: The Neural Network Architectures

Both CBOW and Skip-gram are implemented using shallow neural networks with a single hidden layer. However, they handle their inputs and matrix operations slightly differently.

#### 1. The Skip-Gram Architecture

<p align="center">
  <img src="https://raw.githubusercontent.com/rbg-research/AI-Training/main/text-analytics/chatbots/imgs/skip_gram_net_arch.png" alt="Word2Vec Skip-Gram Neural Network Architecture" width="60%">
</p>
<p align="center"><em>(The Skip-Gram neural network architecture. Image via McCormick ML)</em></p>

1.  **Input Layer:** A single one-hot encoded vector representing the target word (size $V$, where $V$ is vocabulary size).
2.  **Hidden Layer:** A linear layer with $d$ neurons (e.g., 300). **There is no activation function here.** The weight matrix connecting the input to this hidden layer acts as a lookup table. **These weights become the final Word Embeddings.**
3.  **Output Layer:** A softmax classifier predicting a probability distribution for the surrounding context words (size $V$).

#### 2. The CBOW Architecture

<p align="center">
  <img src="https://raw.githubusercontent.com/rbg-research/AI-Training/main/text-analytics/chatbots/imgs/CBOW1.png" alt="Word2Vec CBOW Neural Network Architecture" width="60%">
</p>
<p align="center"><em>(The CBOW neural network architecture. Notice how multiple context words are inputted and averaged.)</em></p>

1.  **Input Layer:** *Multiple* one-hot encoded vectors representing the context words (e.g., 2 words to the left, 2 words to the right).
2.  **Hidden Layer:** The weight matrix acts as a lookup table just like Skip-Gram, extracting a dense vector for *each* input context word. However, because there are multiple inputs, CBOW **aggregates** them (usually by averaging or summing) into a single continuous vector in the hidden layer.
3.  **Output Layer:** A softmax classifier predicting the probability of the single center target word (size $V$).

#### 3. How we train?

<p align="center">
  <img src="https://raw.githubusercontent.com/rbg-research/AI-Training/main/text-analytics/chatbots/imgs/CBOW2.png" alt="Word2Vec CBOW Neural Network Architecture" width="60%">
</p>
<p align="center"><em>(With a window size of 2, skip-gram will generate (up to) four training samples per center word, whereas CBOW only generates one.)</em></p>


Once either network finishes training, we throw away the output layer and keep the hidden layer's weight matrix as our **embedding space**.

#### 4. Embedding Space?

<p align="center">
  <img src="https://raw.githubusercontent.com/rbg-research/AI-Training/main/text-analytics/chatbots/imgs/word2vec2.png" alt="Word2Vec CBOW Neural Network Architecture" width="60%">
</p>
<p align="center"><em>(The significance of these similarities is that the word2vec embedding is somehow reflecting the semantical and syntactical structure of the text corpus.)</em></p>

Let's load a pre-trained Word2Vec model and see if it finally solves the synonym problem we encountered earlier!

In [14]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 52.1 MB/s eta 0:00:00


In [15]:
import gensim.downloader as api
import pandas as pd
import numpy as np

In [16]:
print("Downloading Word2Vec (Google News 300d)... This may take a minute or two.")
# We load a pre-trained model (trained on a massive Google News corpus, 300 dimensions)
# This prevents us from having to train a model from scratch for hours.
w2v_model = api.load('word2vec-google-news-300')

[==================================================] 100.0% 1662.8/1662.8MB downloaded


In [17]:
# Let's extract the dense vectors for our problematic words
movie_vector = w2v_model['movie']
film_vector = w2v_model['film']
terrible_vector = w2v_model['terrible']
excellent_vector = w2v_model['excellent']

print(f"Shape of Word2Vec vectors: {movie_vector.shape}")
print(f"First 5 dimensions of 'movie': {movie_vector[:5].round(3)}")
print(f"First 5 dimensions of 'film':  {film_vector[:5].round(3)}")

Shape of Word2Vec vectors: (300,)
First 5 dimensions of 'movie': [ 0.175 -0.11  -0.2    0.268 -0.064]
First 5 dimensions of 'film':  [-0.004 -0.019 -0.131  0.17   0.105]


In [19]:
# ---------------------------------------------------------
# Word-Level Similarity (Solving the Synonym Problem)
# ---------------------------------------------------------
# Gensim has a built-in similarity function that computes Cosine Similarity
sim_movie_film = w2v_model.similarity('movie', 'film')
sim_movie_terrible = w2v_model.similarity('movie', 'terrible')
sim_excellent_terrible = w2v_model.similarity('excellent', 'terrible')

print("--- Word Similarities (Word2Vec) ---")
print(f"Similarity between 'movie' and 'film': {sim_movie_film:.4f}")
print(f"Similarity between 'movie' and 'terrible': {sim_movie_terrible:.4f}")
print(f"Similarity between 'excellent' and 'terrible': {sim_excellent_terrible:.4f}")

# You can also use it to find the most similar words in the entire vocabulary
print("\n--- Most similar words to 'movie' ---")
display(pd.DataFrame(w2v_model.most_similar('movie', topn=3), columns=["Word", "Cosine Similarity"]))

--- Word Similarities (Word2Vec) ---
Similarity between 'movie' and 'film': 0.8677
Similarity between 'movie' and 'terrible': 0.1662
Similarity between 'excellent' and 'terrible': 0.3301

--- Most similar words to 'movie' ---


,Word,Cosine Similarity
0,film,0.867677
1,movies,0.801311
2,films,0.736301


### Document-Level Similarity (Averaging Word Vectors)

Unlike the Term-Document Matrix or TF-IDF, Word2Vec does not natively create vectors for entire documents. It only knows about individual words.

So, how do we compare two full sentences or documents? The most common baseline approach is **Mean Pooling**. We take the Word2Vec vector for every valid word in the document and average them together to create a single, unified "Document Vector."

Let's write a function to average our word embeddings and compare our toy documents!

In [20]:
# ---------------------------------------------------------
# Document-Level Similarity (Mean Pooling)
# ---------------------------------------------------------

def get_document_vector(text, model):
    """
    Tokenizes the text, looks up the Word2Vec vector for each word,
    and returns the average of those vectors.
    """
    # Simple tokenization (converting to lowercase and splitting by space)
    words = text.lower().split()

    # Collect vectors for words that actually exist in the model's vocabulary
    vectors = [model[word] for word in words if word in model.key_to_index]

    # If the document is empty or has no in-vocabulary words, return a zero vector
    if not vectors:
        return np.zeros(model.vector_size)

    # Calculate the mean across the column axis to get a single average vector
    return np.mean(vectors, axis=0)

# We will reuse the exact same 'corpus' list you defined in Section 1.
# (e.g., corpus = ["the movie was excellent", "the film was excellent", "the movie was terrible"])

# 1. Generate an average Word2Vec vector for every document in your corpus
doc_vectors = np.array([get_document_vector(doc, w2v_model) for doc in corpus])

# 2. Compute the Cosine Similarity matrix for all documents
w2v_sim_matrix = cosine_similarity(doc_vectors)

# 3. Display the results in a clean DataFrame for comparison
doc_labels = [f"Doc {i}" for i in range(len(corpus))]
sim_df = pd.DataFrame(w2v_sim_matrix, index=doc_labels, columns=doc_labels)

print("--- Document-Level Cosine Similarity (Word2Vec Mean Pooling) ---")
print("Notice how the similarity between the 'movie' and 'film' documents")
print("is much higher now compared to the TDM and TF-IDF matrices!\n")
display(sim_df)

--- Document-Level Cosine Similarity (Word2Vec Mean Pooling) ---
Notice how the similarity between the 'movie' and 'film' documents
is much higher now compared to the TDM and TF-IDF matrices!



,Doc 0,Doc 1,Doc 2,Doc 3
Doc 0,1.000000,0.958797,0.827234,0.595009
Doc 1,0.958797,1.000001,0.770139,0.536287
Doc 2,0.827234,0.770139,1.000000,0.849120
Doc 3,0.595009,0.536287,0.849120,1.000000


Under the Hood: Building Word2Vec from Scratch

While libraries like Gensim abstract away the complexity of training embeddings, building the neural network from scratch illuminates exactly how the weight matrices transform into semantic representations.

We will implement both **Skip-Gram** and **Continuous Bag-of-Words (CBOW)** using PyTorch. We will use the same toy corpus from our previous Term-Document Matrix and TF-IDF experiments.

In [21]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

In [22]:
# ---------------------------------------------------------
# 1. Define Corpus and Build Vocabulary
# ---------------------------------------------------------
corpus = [
    "the movie was excellent",
    "the film was excellent",
    "the movie was terrible",
    "the film was terrible"
]

In [23]:
# Tokenize the corpus
tokenized_corpus = [sentence.split() for sentence in corpus]

# Build the vocabulary
vocabulary = set([word for sentence in tokenized_corpus for word in sentence])
vocab_size = len(vocabulary)

# Create mappings from word to index and index to word
word_to_ix = {word: i for i, word in enumerate(vocabulary)}
ix_to_word = {i: word for i, word in enumerate(vocabulary)}

print(f"Vocabulary Size: {vocab_size}")
print(f"Word to Index Map: {word_to_ix}")

Vocabulary Size: 6
Word to Index Map: {'the': 0, 'terrible': 1, 'movie': 2, 'film': 3, 'excellent': 4, 'was': 5}


### The Skip-Gram Implementation

In Skip-Gram, we iterate through our corpus, select a center word, and attempt to predict the context words that surround it within a specific window size ($c$).

```
               ┌───► Context Word 1: 'the'
               │
 Center Word ──┤
  ('movie')    │
               └───► Context Word 2: 'was'

```
We will define a window size of $c=1$ (one word to the left, one word to the right).

In [25]:
# ---------------------------------------------------------
# 2. Generate Skip-Gram Training Data
# ---------------------------------------------------------
WINDOW_SIZE = 1
skipgram_data = []

for sentence in tokenized_corpus:
    for i in range(len(sentence)):
        center_word = sentence[i]

        # Look left and right within the window size
        for j in range(max(0, i - WINDOW_SIZE), min(len(sentence), i + WINDOW_SIZE + 1)):
            if i != j: # Exclude the center word itself
                context_word = sentence[j]
                skipgram_data.append((center_word, context_word))

print(f"First 5 Skip-Gram Pairs (Center, Context): {skipgram_data[:5]}")

First 5 Skip-Gram Pairs (Center, Context): [('the', 'movie'), ('movie', 'the'), ('movie', 'was'), ('was', 'movie'), ('was', 'excellent')]


##### Skip-Gram Neural Network Architecture

In PyTorch, the mathematical equivalent of multiplying a one-hot encoded vector by a weight matrix $\mathbf{W} \in \mathbb{R}^{V \times d}$ is the nn.Embedding layer. This layer serves as our direct lookup table for the word vectors.

In [26]:
# ---------------------------------------------------------
# 3. Define the Skip-Gram Model
# ---------------------------------------------------------
class SkipGramModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(SkipGramModel, self).__init__()
        # Hidden Layer: The Embedding Matrix (V x d)
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)

        # Output Layer: Projects back to Vocabulary Size (d x V)
        self.linear = nn.Linear(embedding_dim, vocab_size)

    def forward(self, center_word_ix):
        # 1. Extract the dense vector for the center word
        embeds = self.embeddings(center_word_ix)

        # 2. Project back to vocabulary space
        out = self.linear(embeds)

        # 3. Apply Log Softmax to get probability distribution
        log_probs = F.log_softmax(out, dim=1)
        return log_probs

# Hyperparameters
EMBEDDING_DIM = 2 # Keeping it small to visualize later if needed
EPOCHS = 100
LEARNING_RATE = 0.01

skipgram_model = SkipGramModel(vocab_size, EMBEDDING_DIM)
loss_function = nn.NLLLoss() # Negative Log Likelihood Loss
optimizer = optim.SGD(skipgram_model.parameters(), lr=LEARNING_RATE)

# ---------------------------------------------------------
# 4. Train the Skip-Gram Model
# ---------------------------------------------------------
print("--- Training Skip-Gram ---")
for epoch in range(EPOCHS):
    total_loss = 0
    for center, context in skipgram_data:
        # Step 1: Prepare the inputs (convert words to tensor indices)
        center_tensor = torch.tensor([word_to_ix[center]], dtype=torch.long)
        context_tensor = torch.tensor([word_to_ix[context]], dtype=torch.long)

        # Step 2: Zero the gradients
        skipgram_model.zero_grad()

        # Step 3: Forward pass
        log_probs = skipgram_model(center_tensor)

        # Step 4: Compute loss and backpropagate
        loss = loss_function(log_probs, context_tensor)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss:.4f}")

# The hidden layer weights are now our embeddings!
print("\nSkip-Gram Embeddings for 'movie':")
print(skipgram_model.embeddings.weight[word_to_ix['movie']].detach().numpy())

--- Training Skip-Gram ---
Epoch 20/100 | Loss: 29.8310
Epoch 40/100 | Loss: 25.2066
Epoch 60/100 | Loss: 23.2051
Epoch 80/100 | Loss: 22.1571
Epoch 100/100 | Loss: 21.5254

Skip-Gram Embeddings for 'movie':
[-0.22526997  2.1434152 ]


### The CBOW Implementation
Continuous Bag-of-Words flips the paradigm. It takes a list of surrounding context words, aggregates them, and predicts the single center word.

```
Context Word 1 ───┐
    ('the')        │
                   ├───► [ Aggregation Layer ] ───► Center Word
                   │       (Sum or Average)          ('movie')
 Context Word 2 ───┘
    ('was')
```



In [27]:
# ---------------------------------------------------------
# 5. Generate CBOW Training Data
# ---------------------------------------------------------
cbow_data = []

for sentence in tokenized_corpus:
    for i in range(WINDOW_SIZE, len(sentence) - WINDOW_SIZE):
        context_words = [sentence[i - 1], sentence[i + 1]] # c=1 left and right
        target_word = sentence[i]
        cbow_data.append((context_words, target_word))

print(f"First 3 CBOW Pairs (Contexts, Center): {cbow_data[:3]}")

First 3 CBOW Pairs (Contexts, Center): [(['the', 'was'], 'movie'), (['movie', 'excellent'], 'was'), (['the', 'was'], 'film')]


In [28]:
# ---------------------------------------------------------
# 6. Define the CBOW Model
# ---------------------------------------------------------
class CBOWModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(CBOWModel, self).__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.linear = nn.Linear(embedding_dim, vocab_size)

    def forward(self, context_word_indices):
        # 1. Extract embeddings for all context words
        embeds = self.embeddings(context_word_indices)

        # 2. Aggregate contexts (Summing them along the sequence dimension)
        # Reshaping to (1, -1) to match the expected batch dimension
        aggregated_embeds = torch.sum(embeds, dim=0).view(1, -1)

        # 3. Project to vocabulary space
        out = self.linear(aggregated_embeds)

        # 4. Log Softmax for probabilities
        log_probs = F.log_softmax(out, dim=1)
        return log_probs

cbow_model = CBOWModel(vocab_size, EMBEDDING_DIM)
optimizer_cbow = optim.SGD(cbow_model.parameters(), lr=LEARNING_RATE)

# ---------------------------------------------------------
# 7. Train the CBOW Model
# ---------------------------------------------------------
print("\n--- Training CBOW ---")
for epoch in range(EPOCHS):
    total_loss = 0
    for context, target in cbow_data:
        # Step 1: Prepare inputs
        context_idxs = torch.tensor([word_to_ix[w] for w in context], dtype=torch.long)
        target_tensor = torch.tensor([word_to_ix[target]], dtype=torch.long)

        # Step 2: Zero gradients
        cbow_model.zero_grad()

        # Step 3: Forward pass
        log_probs = cbow_model(context_idxs)

        # Step 4: Compute loss and backpropagate
        loss = loss_function(log_probs, target_tensor)
        loss.backward()
        optimizer_cbow.step()

        total_loss += loss.item()

    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss:.4f}")

# Extract final embeddings
print("\nCBOW Embeddings for 'film':")
print(cbow_model.embeddings.weight[word_to_ix['film']].detach().numpy())


--- Training CBOW ---
Epoch 20/100 | Loss: 7.7676
Epoch 40/100 | Loss: 4.6713
Epoch 60/100 | Loss: 3.7069
Epoch 80/100 | Loss: 3.3674
Epoch 100/100 | Loss: 3.2128

CBOW Embeddings for 'film':
[-1.5986036 -1.2790265]


### Evaluating Learned Word Similarities

Now that both models have trained on our tiny corpus, let's extract their embedding weights and calculate the **Cosine Similarity** between specific word pairs:

$\text{Cosine Similarity}(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\|_2 \|\mathbf{v}\|_2}$

We expect `'movie'` and `'film'` to exhibit high cosine similarity because they appeared in identical context windows during training.

In [29]:
def compute_pair_similarity(word1, word2, model):
    """
    Looks up the learned embedding vectors from the model's
    embedding layer and computes cosine similarity.
    """
    if word1 not in word_to_ix or word2 not in word_to_ix:
        raise ValueError(f"One or both words not in vocabulary.")

    # Convert words to tensor indices
    idx1 = torch.tensor([word_to_ix[word1]], dtype=torch.long)
    idx2 = torch.tensor([word_to_ix[word2]], dtype=torch.long)

    # Extract the weight vectors from the nn.Embedding layer
    vec1 = model.embeddings(idx1)
    vec2 = model.embeddings(idx2)

    # Calculate cosine similarity using PyTorch functional API
    similarity = F.cosine_similarity(vec1, vec2)
    return similarity.item()

# Word pairs to evaluate
word_pairs = [
    ("movie", "film"),         # Synonyms (should be high)
    ("movie", "terrible"),     # Target vs Context
    ("excellent", "terrible")  # Opposite sentiment
]

# Build comparison table
results = []
for w1, w2 in word_pairs:
    sg_sim = compute_pair_similarity(w1, w2, skipgram_model)
    cbow_sim = compute_pair_similarity(w1, w2, cbow_model)

    results.append({
        "Word Pair": f"'{w1}' vs '{w2}'",
        "Skip-Gram Sim": f"{sg_sim:.4f}",
        "CBOW Sim": f"{cbow_sim:.4f}"
    })

# Display comparison matrix
print("--- Scratch Word2Vec Similarity Results ---")
df_results = pd.DataFrame(results)
display(df_results)

--- Scratch Word2Vec Similarity Results ---


,Word Pair,Skip-Gram Sim,CBOW Sim
0,'movie' vs 'film',0.9989,0.8149
1,'movie' vs 'terrible',-0.0682,0.2235
2,'excellent' vs 'terrible',0.8518,-0.8936


Looking at the output from our custom PyTorch implementation:

| Word Pair | Skip-Gram Similarity | CBOW Similarity |
| :--- | :--- | :--- |
| `'movie'` vs `'film'` | 0.9989 | 0.8149 |
| `'movie'` vs `'terrible'` | -0.0682 | 0.2235 |
| `'excellent'` vs `'terrible'`| 0.8518 | -0.8936 |

**Observations from our toy models:**
*   **Synonyms Succeed:** Both architectures successfully recognized that `'movie'` and `'film'` are nearly identical concepts because they share the exact same context words in our tiny corpus.
*   **The Antonym Quirk:** Notice how Skip-Gram gave a high similarity (0.8518) to `'excellent'` and `'terrible'`. This is a famous characteristic of distributional semantics! Because antonyms frequently appear in the exact same grammatical position (e.g., "The movie was ___"), Word2Vec often plots them close together in the vector space.

---

### What did we solve?

Stepping back to look at Word2Vec as a whole (like the pre-trained Gensim model we used earlier), we have successfully addressed the major flaws of count-based models:

1.  **The Synonym Problem:** The similarity between "movie" and "film" in our Gensim model was roughly **0.86**. In our previous TF-IDF model, the similarity between these words was exactly 0.0000 because they never appeared in the exact same document in our tiny corpus. By using a large corpus and dense embeddings, the model successfully understands that these words are highly related concepts.
2.  **Extreme Sparsity:** Instead of vectors being filled with tens of thousands of zeros to match the entire vocabulary size, the word "movie" is now completely represented by a dense, memory-efficient 300-dimensional vector of real numbers.
3.  **Semantic Geometry:** Because meaning is distributed across the vector space, we can perform linear algebra on text (e.g., discovering analogies like *King - Man + Woman = Queen*).

### What problems still persist?

While Word2Vec is a massive leap forward, it still has architectural limitations:

1.  **Polysemy (Multiple Meanings):** Word2Vec creates exactly **one** static vector per word. It cannot distinguish between "bank" (a financial institution) and "bank" (the side of a river). It averages all meanings into a single, blended representation.
2.  **Out-of-Vocabulary (OOV) Words:** If a word was not in the training corpus, the model completely fails to process it. It cannot infer the meaning of a newly coined word, a prefix, or a typo.

To fix the Out-of-Vocabulary problem, we need to look below the word level. In the next session, we will explore **GloVe** and sub-word models like **FastText**.